# 🎬 MoneyPrinterTurbo — Colab Server

This notebook spins up the MPT API server and exposes it via ngrok.

**Step 1:** Fill in your API keys below
**Step 2:** Run all cells
**Step 3:** Copy the `MPT_BASE_URL` at the bottom and paste it into your `.env.local`

⚠️ **Runtime:** Use **GPU** runtime (Runtime → Change runtime type → GPU T4)
⏱️ **Approximate time:** 3–5 minutes to fully start

## 1. 🔑 API Keys — FILL THESE IN

Get them here:
- **Gemini API key:** https://aistudio.google.com/app/apikey (free tier available)
- **Pexels API key:** https://www.pexels.com/api/ (free tier: 200 credits/month)
- **ngrok auth token:** https://dashboard.ngrok.com/get-started/your-authtoken (free account)

In [ ]:
# ↓ REPLACE THESE WITH YOUR ACTUAL KEYS
GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE"   # e.g. AIza...
PEXELS_API_KEY = "YOUR_PEXELS_API_KEY_HERE"   # e.g. 5634926...
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"         # e.g. 2abc...xyz

## 2. Install & Clone

Clones MPT, installs FFmpeg, and sets up the Python environment.

In [ ]:
# Install system deps
!apt-get update -qq && apt-get install -y -qq ffmpeg git curl

# Clone MoneyPrinterTurbo (use a pinned release to keep it stable)
!git clone --depth 1 --branch v2.4.1 https://github.com/harry0706/MoneyPrinterTurbo.git /content/MoneyPrinterTurbo 2>&1 | tail -5

%cd /content/MoneyPrinterTurbo

# Install Python deps
!pip install -q -r requirements.txt 2>&1 | tail -3

## 3. Write config.toml

Configures MPT with your API keys. Uses Gemini for LLM + Pexels for stock footage.

In [ ]:
import os

config_toml = f"""
# MoneyPrinterTurbo — Auto-generated by Colab notebook
# Generated at: {os.popen('date').read().strip()}

log_level = "INFO"
listen_host = "0.0.0.0"
listen_port = 8080

[app]
video_source = "pexels"
pexels_api_keys = [\"{PEXELS_API_KEY}\"]

[llm]
llm_provider = "gemini"
gemini_api_key = \"{GEMINI_API_KEY}\"

[ui]
hide_log = true
"""

with open("config.toml", "w") as f:
    f.write(config_toml)

print("✅ config.toml written")

## 4. Install ngrok

Exposes the local server to the internet so your SaaS can reach it.

In [ ]:
!curl -s https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null && echo "deb https://ngrok-agent.s3.amazonaws.com/bin stable main" | tee /etc/apt/sources.list.d/ngrok.list >/dev/null && apt-get update -qq && apt-get install -y -qq ngrok

# Authenticate ngrok
!ngrok config add-authtoken {NGROK_TOKEN}

print("✅ ngrok installed and authenticated")

## 5. Start MPT server

Starts the FastAPI server in the background. Keep this cell running.

In [ ]:
import subprocess
import time

# Start MPT in background
mpt_process = subprocess.Popen(
    ["python", "main.py"],
    cwd="/content/MoneyPrinterTurbo",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give it time to start
print("Starting MPT server...")
time.sleep(8)

# Check if it's up
import urllib.request
try:
    resp = urllib.request.urlopen("http://127.0.0.1:8080/docs", timeout=5)
    print(f"✅ MPT server running on port 8080 (status: {resp.status})")
except Exception as e:
    print(f"⚠️ Server may still be starting... ({e})")
    print("Waiting 5 more seconds...")
    time.sleep(5)

## 6. Start ngrok tunnel

Exposes port 8080 publicly. The URL this produces is your `MPT_BASE_URL`.

In [ ]:
import subprocess
import time
import re

# Start ngrok tunnel on port 8080
ngrok_process = subprocess.Popen(
    ["ngrok", "http", "8080", "--log", "stdout"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Waiting for ngrok to establish tunnel...")
time.sleep(5)

# Poll ngrok for the public URL
import urllib.request
mpt_base_url = None
for attempt in range(12):
    try:
        resp = urllib.request.urlopen("http://127.0.0.1:4040/api/tunnels", timeout=5)
        import json
        tunnels = json.loads(resp.read())
        for tunnel in tunnels.get("tunnels", []):
            if tunnel.get("proto") == "https":
                mpt_base_url = tunnel["public_url"]
                break
        if mpt_base_url:
            break
    except:
        pass
    time.sleep(2)

if mpt_base_url:
    print(f"\n🎉 MPT server is live!")
    print(f"\n📋 Add this to your .env.local:")
    print(f"   MPT_BASE_URL={mpt_base_url}")
    print(f"\n🌐 API docs: {mpt_base_url}/docs")
    print(f"\n⚠️ Keep this cell running! Close it and the tunnel dies.")
else:
    print("❌ Could not get ngrok URL. Check your auth token.")

## 7. Quick smoke test

Posts a test job to make sure everything is connected.

In [ ]:
import urllib.request
import json

if not mpt_base_url:
    print("⚠️ No ngrok URL yet — run cell 6 first")
else:
    payload = json.dumps({
        "video_subject": "Why AI will change video creation forever",
        "video_aspect": "9:16",
        "voice_name": "gemini:Zephyr"
    }).encode()
    
    req = urllib.request.Request(
        f"{mpt_base_url}/videos",
        data=payload,
        headers={"Content-Type": "application/json"}
    )
    
    try:
        resp = urllib.request.urlopen(req, timeout=10)
        result = json.loads(resp.read())
        task_id = result.get("data", {}).get("task_id") or result.get("task_id")
        print(f"✅ Job submitted! task_id: {task_id}")
        print(f"   Poll at: {mpt_base_url}/tasks/{task_id}")
    except Exception as e:
        print(f"❌ Test failed: {e}")
        print("   Check your API keys in config.toml")

---

## 📋 To use in your SaaS

1. Copy the `MPT_BASE_URL` from cell 6
2. Add it to your Vercel env vars: **Settings → Environment Variables**
3. Deploy MPTSAAS to Vercel
4. Test the dashboard — hit **Generate video**

**Note:** ngrok free tier tunnels die when this Colab session ends.
For production, deploy MPT to **Modal**, **RunPod**, or a GPU VPS.